In [ ]:
# Imports
import os
from si_units import *  # SI units package (separate from feos in v0.8.0)
from feos.pcsaft import PcSaftParameters, PcSaftRecord, Identifier, PureRecord
from feos.eos import EquationOfState, State, PhaseDiagram
from feos.eos import Contributions
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

from data_tools.utils import generate_state_points
# from data_tools.utils_before_full7 import generate_state_points

print("Imports successful!")

msize=12
mwidth = 2
fsize=16
alpha=0.3
lsize = 2
framewidth = 3
markeredgewidth=3
markeredgewidth_no=10

In [ ]:
csv_path = "SI_pcp-saft_parameters.csv"

saft_data = pd.read_csv(csv_path, sep="\t")
saft_data = saft_data.fillna(0.0)

saft_data.shape

In [ ]:
keys = [  'm', 'sigma',
       'epsilon_k', 'mu', 'kappa_ab', 'epsilon_k_ab' ]

for key in keys:
    col = np.array(saft_data[key])
    col_norm = ( col - col.min() ) / ( col.max() - col.min() )
    saft_data[key+"_norm"] = col_norm

In [ ]:
dataset_path = "full_data_6/"
rho_gas_factor = [200,0.03]
rho_liq_factor = [200,0.03]
filter_two_phase=True
n_isotherms=50
n_isochores=50
T_factor=1.2
P_factor=20.0
T_min_factor=0.4
n_low_density_cols=0
rho_min_factor=0.7
ideal_gas_rho_norm_threshold = None

try:
    os.mkdir(dataset_path)
except:
    None


saft_path = dataset_path + "saft_parameters.csv"

saft_data["ID"] = np.arange(saft_data.shape[0])
saft_data.to_csv(saft_path, index=False)
saft_data = pd.read_csv(saft_path)

In [ ]:
saft_data

In [ ]:
plt.plot( saft_data.mu )

In [ ]:
smiles = "C"
smiles = "CCCC"
#smiles = "CCCCCC"
smiles = "O"

state_df, vle_df, critical_point = generate_state_points(
    smiles, saft_path,
    n_isotherms=n_isotherms,
    n_isochores=n_isochores,
    T_factor=T_factor,
    P_factor=P_factor,
    T_min_factor=T_min_factor,
    filter_two_phase=filter_two_phase,      
    rho_gas_factor=rho_gas_factor,         
    rho_liq_factor=rho_liq_factor,
    n_low_density_cols=n_low_density_cols,
    ideal_gas_rho_norm_threshold=ideal_gas_rho_norm_threshold,
    keep_negative_two_phase=keep_negative_two_phase,
)



plt.plot( state_df.molar_density_mol_m3 /1000  , state_df.temperature_K ,  ".", color="cornflowerblue",label="training data" )
plt.plot( vle_df.molar_density_vapor_mol_m3 /1000   , vle_df.temperature_K ,  "k-", linewidth=lsize, label="VLE" )
plt.plot( vle_df.molar_density_liquid_mol_m3 /1000   , vle_df.temperature_K ,  "k-", linewidth=lsize )

vle_df = vle_df[ vle_df.temperature_K < vle_df.temperature_K.max() - 30 ]

#plt.plot( vle_df.molar_density_vapor_mol_m3   , vle_df.temperature_K ,  "k--" )
#plt.plot( vle_df.molar_density_liquid_mol_m3   , vle_df.temperature_K ,  "k--" )

plt.xlabel(r"$\rho~/~kmol/m^3$",fontsize=fsize)
plt.ylabel(r"temperature / K",fontsize=fsize)

plt.legend(fontsize=fsize, frameon=False)

plt.xticks(fontsize=fsize)  
plt.yticks(fontsize=fsize)  

plt.savefig(f"data_{smiles}_generation.png",bbox_inches='tight')
plt.savefig(f"data_{smiles}_generation.pdf",bbox_inches='tight')


print("min vle density",vle_df.molar_density_vapor_mol_m3.min())
print("min temperature",vle_df.temperature_K.min())
print( state_df.shape, vle_df.shape )

In [ ]:
state_df.pressure_Pa.max() / 1e9

In [ ]:
p = state_df.molar_density_mol_m3 < 10000

plt.plot( state_df.molar_density_mol_m3[p]   , state_df.temperature_K[p] ,  "." )

In [ ]:
plt.plot( state_df.molar_density_mol_m3   , state_df.pressure_Pa ,  "o" )


In [ ]:
cmap = plt.scatter( state_df.molar_density_mol_m3   , state_df.temperature_K , c=state_df.pressure_Pa )

dummy = state_df[state_df.pressure_Pa <= 1]
plt.plot( dummy.molar_density_mol_m3   , dummy.temperature_K, ".r" )

plt.colorbar(cmap, )

In [ ]:
state_dfs = []
vle_dfs = []
for i,row in saft_data.iterrows():
    print(i , "--" , smiles)
    smiles = row["canonical_smiles"]
    state_df, vle_df, critical_point = generate_state_points(
        smiles, saft_path,
        n_isotherms=n_isotherms,
        n_isochores=n_isochores,
        T_factor=T_factor,
        P_factor=P_factor,
        T_min_factor=T_min_factor,
        filter_two_phase=filter_two_phase,      
        rho_gas_factor=rho_gas_factor,         
        rho_liq_factor=rho_liq_factor,
        n_low_density_cols=n_low_density_cols,
        ideal_gas_rho_norm_threshold=ideal_gas_rho_norm_threshold,
        keep_negative_two_phase=keep_negative_two_phase,
    )
    state_df["ID"] = row["ID"]
    state_df["canonical_smiles"] = smiles
    state_dfs.append(state_df)

    vle_df = vle_df[ vle_df.temperature_K < vle_df.temperature_K.max() - 30 ]
    vle_df["ID"] = row["ID"]
    vle_df["canonical_smiles"] = smiles
    vle_dfs.append(vle_df)

    # break

states_df = pd.concat(state_dfs)
vles_df = pd.concat(vle_dfs)

In [ ]:
states_df.hist(column="temperature_K", bins=100)

In [ ]:
states_df.hist(column="molar_density_mol_m3",bins=100)

In [ ]:
states_df.hist(column="pressure_Pa",bins=200)

In [ ]:
states_df.hist(column="pressure_res_Pa",bins=100)

In [ ]:
states_df.shape, vles_df.shape

In [ ]:
states_df.to_csv( dataset_path+"data_res_full.csv", index=False )
vles_df.to_csv( dataset_path+"data_vle_full.csv", index=False )

In [ ]:
dataset_path